In [1]:
%load_ext autoreload
%autoreload 2

# Imports

In [2]:
from pathlib import Path
import shutil

import torch
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import open3d as o3d
from torch import Tensor, nn
import MinkowskiEngine as ME
import faiss
from tqdm import tqdm


from opr.models.place_recognition import MinkLoc3D, MinkLoc3Dv2
from opr.pipelines.place_recognition import PlaceRecognitionPipeline

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


/usr/local/lib/python3.10/dist-packages/MinkowskiEngine-0.5.4-py3.10-linux-x86_64.egg/MinkowskiEngine/__init__.py:36: UserWarning: The environment variable `OMP_NUM_THREADS` not set. MinkowskiEngine will automatically set `OMP_NUM_THREADS=16`. If you want to set `OMP_NUM_THREADS` manually, please export it on the command line before running a python script. e.g. `export OMP_NUM_THREADS=12; python your_program.py`. It is recommended to set it below 24.
  warnings.warn(
2025-08-25 18:09:21.567 | WARNING  | opr.models.place_recognition.pointmamba:<module>:16 - The 'pointmamba' package is not installed. Please install it manually if neccessary.
Process ForkProcess-31:
Process ForkProcess-14:
Process ForkProcess-7:
Process ForkProcess-2:
Process ForkProcess-6:
Process ForkProcess-28:
Process ForkProcess-30:
Process ForkProcess-13:
Process ForkProcess-19:
Process ForkProcess-3:
Process ForkProcess-1:
Process ForkProcess-23:
Process ForkProcess-5:
Process ForkProcess-10:
Process ForkProcess-1

# Constants

In [3]:
REPO_ROOT = Path("/home/docker_mmpr/multimodal-place-recognition")
DATASETS_ROOT = Path("/home/docker_mmpr/Datasets/")
LOCAL_DATA_DIR = Path("/home/docker_mmpr/multimodal-place-recognition/data/2025-03-26-mmpr-datasets/")

SBER_OFFICE_DATA_DIR = DATASETS_ROOT / "2025-03-26-mmpr-datasets" / "keyframe-lidar-maps" / "keyframe-lidar-maps" /"mmpr_dataset" / "map1" / "keyframe_map"
assert SBER_OFFICE_DATA_DIR.exists(), f"Data directory {SBER_OFFICE_DATA_DIR} does not exist."


# DataReader definition

In [4]:
from scipy.spatial.transform import Rotation as R

class DataReader:
    def __init__(
            self, csv_file: str | Path, lidar_scans_dir: str | Path, pointcloud_quantization_size: float = 0.1, agg_thres: float = 0.0
        ) -> None:
        """Initialize DataReader for pose-timestamped point cloud data.

        Args:
            csv_file (str | Path): Path to the CSV file containing pose and timestamp data.
            lidar_scans_dir (str | Path): Directory containing the lidar scans in PCD format.
            pointcloud_quantization_size (float): Size for quantizing the point cloud coordinates.
                Default is 0.1.
        Raises:
            FileNotFoundError: If the CSV file or lidar scans directory does not exist.
        """
        self.agg_thres = agg_thres 
        
        csv_file = Path(csv_file)
        if not csv_file.exists():
            raise FileNotFoundError(f"CSV file {csv_file} does not exist.")

        self.lidar_scans_dir = Path(lidar_scans_dir)
        if not self.lidar_scans_dir.exists():
            raise FileNotFoundError(f"Lidar scans directory {self.lidar_scans_dir} does not exist.")

        self.df = self.read_csv(csv_file)

        self.scans_list = []
        for scan_id in self.df['lidar_timestamp'].values:
            path = self.lidar_scans_dir / f"{scan_id:06d}.pcd"
            if not path.exists():
                raise FileNotFoundError(f"Missing scan file: {path}")
            self.scans_list.append(path)

        self._pointcloud_quantization_size = pointcloud_quantization_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        if self.agg_thres > 0:
            return self._get_aggregated_item(idx)
        return self._get_single_item(idx)
    
    def _get_single_item(self, idx):
        """Get single scan without aggregation"""
        pose = self.df[["x", "y", "z", "qx", "qy", "qz", "qw"]].iloc[idx].to_numpy()
        scan_filepath = self.scans_list[idx]
        pc_coords, pc_feats = self.read_scan(scan_filepath)

        output_dict = {
            "pose": Tensor(pose),
            "pointcloud_lidar_coords": Tensor(pc_coords),
            "pointcloud_lidar_feats": Tensor(pc_feats)
        }

        return output_dict
    
    def _get_aggregated_item(self, idx):
        """Get aggregated scans within threshold distance"""
        scan_filepath = self.scans_list[idx]
        current_coords, current_feats = self.read_scan(scan_filepath)
        current_pose = self.df[["x", "y", "z", "qx", "qy", "qz", "qw"]].iloc[idx].to_numpy()
        current_loc = current_pose[:3]
        
        # Find nearby scans to aggregate
        agg_points = []
        agg_feats = []
        for i in range(max(0, idx-100), idx+1):  # Look back up to 100 frames
            pose = self.df[["x", "y", "z", "qx", "qy", "qz", "qw"]].iloc[i].to_numpy()
            if np.linalg.norm(pose[:3] - current_loc) <= self.agg_thres:
                # Transform points to current frame
                T_current = self.pose_to_matrix(*current_pose)
                T_other = self.pose_to_matrix(*pose)
                T_other_to_current = np.linalg.inv(T_current) @ T_other
                
                scan_filepath = self.scans_list[i]
                points, feats = self.read_scan(scan_filepath)
                points_transformed = (T_other_to_current[:3, :3] @ points.T).T + T_other_to_current[:3, 3]
                agg_points.append(points_transformed)

                agg_feats.append(feats)
        
        if agg_points:
            agg_points = np.concatenate(agg_points, axis=0)
            agg_feats = np.concatenate(agg_feats, axis=0)
        else:
            agg_points = current_coords
            agg_feats = current_feats

        agg_points = np.ascontiguousarray(agg_points)
        agg_feats = np.ascontiguousarray(agg_feats)
        # agg_feats = np.ones((agg_points.shape[0], 1))
        
        return {
            "pointcloud_lidar_coords": torch.tensor(agg_points, dtype=torch.float32),
            "pointcloud_lidar_feats": torch.tensor(agg_feats, dtype=torch.float32),
            "pose": torch.tensor(current_pose, dtype=torch.float32),
        }

    @staticmethod
    def pose_to_matrix(tx, ty, tz, qx, qy, qz, qw):
        rot = R.from_quat([qx, qy, qz, qw]).as_matrix()
        T = np.eye(4)
        T[:3, :3] = rot
        T[:3, 3] = [tx, ty, tz]
        return T

    def collate_fn(self, batch: list[dict[str, Tensor]]) -> dict[str, Tensor]:
        """Collate function to combine a batch of data points into a single dictionary.
        Args:
            batch (list[dict[str, Tensor]]): A list of dictionaries containing pose and point cloud data.
        Returns:
            dict: A dictionary containing:
                - poses (Tensor): Poses as an Nx7 tensor.
                - pointclouds_lidar_coords (Tensor): Point cloud coordinates as an Nx3 tensor.
                - pointclouds_lidar_feats (Tensor): Point cloud features as an Nx1 tensor.
        """
        poses = torch.stack([item['pose'] for item in batch])

        coords_list = [e["pointcloud_lidar_coords"] for e in batch]
        feats_list = [e["pointcloud_lidar_feats"] for e in batch]
        quantized_coords_list = []
        quantized_feats_list = []
        for coords, feats in zip(coords_list, feats_list):
            quantized_coords, quantized_feats = ME.utils.sparse_quantize(
                coordinates=coords,
                features=feats,
                quantization_size=self._pointcloud_quantization_size,
            )
            quantized_coords_list.append(quantized_coords)
            quantized_feats_list.append(quantized_feats)

        return {
            "poses": poses,
            "pointclouds_lidar_coords": ME.utils.batched_coordinates(quantized_coords_list),
            "pointclouds_lidar_feats": torch.cat(quantized_feats_list)
        }

    def read_scan(self, scan_filepath: str | Path) -> tuple[np.ndarray, np.ndarray]:
        """Read a point cloud scan from a file.
        Args:
            scan_filepath (str | Path): Path to the point cloud file.
        Returns:
            tuple: A tuple containing:
                - coordinates (np.ndarray): The coordinates of the point cloud as an Nx3 array.
                - features (np.ndarray): The features of the point cloud as an Nx1 array (intensity or ones).
        Raises:
            ValueError: If the scan file is empty or has an unexpected format.
        """
        scan = o3d.io.read_point_cloud(str(scan_filepath))
        if not scan.has_points():
            raise ValueError(f"Scan file {scan_filepath} is empty or invalid.")
        # Convert to numpy array for easier manipulation
        scan = np.asarray(scan.points)
        coordinates = scan[:, :3]  # Get the first three columns (x, y, z)
        if scan.shape[1] == 3:
            features = np.ones((coordinates.shape[0], 1))
        elif scan.shape[1] == 4:
            features = scan[:, 3:4]  # Get the fourth column (intensity)
        else:
            raise ValueError(f"Unexpected scan format with shape {scan.shape}. Expected 3 or 4 columns.")
        return coordinates, features

    def read_csv(self, filepath: str | Path) -> pd.DataFrame:
        """Read a CSV file containing pose and timestamp data.
        Args:
            filepath (str | Path): Path to the CSV file.
        Returns:
            pd.DataFrame: A DataFrame containing the pose and timestamp data.
        Raises:
            FileNotFoundError: If the CSV file does not exist.
        """
        dtype_mapping = {
            'pose_timestamp': np.int64,
            'lidar_timestamp': np.int64,
            'x': np.float64,
            'y': np.float64,
            'z': np.float64,
            'qx': np.float64,
            'qy': np.float64,
            'qz': np.float64,
            'qw': np.float64,
        }
        df = pd.read_csv(filepath, dtype=dtype_mapping)
        return df


# Init model

In [5]:
# weights = torch.load(REPO_ROOT / "data" / "checkpoints" / "minkloc3dv2_baseline.pth")
# weights = torch.load(REPO_ROOT / "data" / "checkpoints" / "minkloc3d_nclt.pth")
# weights = torch.load(REPO_ROOT / "data" / "finetuned_checkpoints" / "v1" / "last.pth")
# weights = torch.load(REPO_ROOT / "data" / "finetuned_checkpoints" / "v1" / "best.pth")
weights = torch.load(REPO_ROOT / "data" / "checkpoints" / "minkloc3dv2_nclt.pth")

# model = MinkLoc3Dv2()
model = MinkLoc3D()
model.load_state_dict(weights, strict=False)
model.eval()
if torch.cuda.is_available():
    model = model.cuda()
else:
    print("CUDA is not available, running on CPU.")

# Build Faiss index

In [6]:
from copy import deepcopy

class TopKPRPipeline(PlaceRecognitionPipeline):
    def infer_top_k(self, input_data: dict[str, torch.Tensor], top_k: int = 5) -> dict[str, np.ndarray]:
        input_data = self._preprocess_input(input_data)
        output = {}
        with torch.no_grad():
            descriptor = self.model(input_data)["final_descriptor"].cpu().numpy().reshape(1, -1)

        _, predictions = self.database_index.search(descriptor, k=top_k)
        pred_ids = deepcopy(predictions[0]) # maybe tolist?
        pred_poses = self.database_df.iloc[pred_ids][['x', 'y', 'z', 'qx', 'qy', 'qz', 'qw']].to_numpy(dtype=float)
        output["idx"] = pred_ids
        output["pose"] = pred_poses
        output["descriptor"] = descriptor[0]
        return output

In [7]:
def eval_model(query_reader, database_dl, model):
    descriptors_list = []
    with torch.no_grad():
        for batch in tqdm(database_dl):
            batch = {k: v.to("cuda") for k, v in batch.items()}
            descriptors = model(batch)["final_descriptor"]
            descriptors_list.append(descriptors)
    descriptors = torch.cat(descriptors_list, dim=0)
    print(f"Descriptors shape: {descriptors.shape}")
    
    # Create L2 distance FAISS index for nearest neighbor search
    faiss_index = faiss.IndexFlatL2(descriptors.shape[1])
    faiss_index.add(descriptors.cpu().numpy())
    faiss.write_index(
        faiss_index,
        str(REPO_ROOT / "data" / "2025-03-26-mmpr-datasets" / "mmpr_dataset_1_database" / "index.faiss")
    )
    
    # Copy pose data as track.csv (required by PlaceRecognitionPipeline)
    shutil.copy(
        LOCAL_DATA_DIR / "db_lidar_frames.csv",
        REPO_ROOT / "data" / "2025-03-26-mmpr-datasets" / "mmpr_dataset_1_database" / "track.csv"
    )

    pipeline = TopKPRPipeline(
        database_dir=REPO_ROOT / "data" / "2025-03-26-mmpr-datasets" / "mmpr_dataset_1_database",
        model=model,
        device="cuda",
        pointcloud_quantization_size=PC_QUANTIZATION_SIZE,
    )
    # Evaluate place recognition accuracy by comparing retrieved vs ground truth poses
    translation_errors = []
    rotation_errors = []  # angle in radians
    
    top_k_candidates = 5
    distance_threshold = 5.0  # meters
    
    recalls = [0] * top_k_candidates
    
    for q_idx, query in tqdm(enumerate(query_reader), total=len(query_reader)):
        query = {k: v.to("cuda") for k, v in query.items()}
        results = pipeline.infer_top_k(query, top_k=top_k_candidates)
        db_idx = results["idx"]  # Retrieved database indices, now it is a list
        q_pose = query["pose"].cpu().numpy()  # Ground truth query pose
    
        q_loc, q_rot = q_pose[:3], q_pose[3:]
    
        match_at_k = [False] * top_k_candidates
    
        best_translation_error, best_rotation_error = float('inf'), float('inf')
        for rank, db_pose in enumerate(results["pose"]): # Retrieved database poses, now it is a list
            db_loc, db_rot = db_pose[:3], db_pose[3:]
            translation_error = np.linalg.norm(q_loc - db_loc)
            rotation_error = 2 * np.arccos(np.abs(np.dot(q_rot, db_rot)))
    
            best_translation_error = min(best_translation_error, translation_error)
            best_rotation_error = min(best_rotation_error, rotation_error)
    
            if translation_error < distance_threshold:
                match_at_k[rank:] = [True] * (len(match_at_k) - rank)
                break
    
        translation_errors.append(best_translation_error)
        rotation_errors.append(best_rotation_error)
    
        for k in range(top_k_candidates):
            recalls[k] += match_at_k[k]
    
    translation_errors = np.array(translation_errors)
    rotation_errors = np.array(rotation_errors)
    
    print(f"Mean translation error: {np.mean(translation_errors):.2f} m")
    print(f"Mean rotation error: {np.rad2deg(np.mean(rotation_errors)):.2f} degrees")
    
    print(f"Median translation error: {np.median(translation_errors):.2f} m")
    print(f"Median rotation error: {np.rad2deg(np.median(rotation_errors)):.2f} degrees")
    
    total = len(query_reader)
    for k in range(top_k_candidates):
        recall_at_k = recalls[k] / total
        print(f"Recall@{k + 1} at {distance_threshold:.1f} m: {recall_at_k:.2%}")

In [8]:
import gc
import torch

PC_QUANTIZATION_SIZE = 0.05

for agg_thres in range(5):
    # Database: batched processing for efficient descriptor extraction
    database_reader = DataReader(
        csv_file=LOCAL_DATA_DIR / "db_lidar_frames.csv",
        lidar_scans_dir=SBER_OFFICE_DATA_DIR / "scans",
        pointcloud_quantization_size=PC_QUANTIZATION_SIZE,
        agg_thres=agg_thres,
    )
    
    database_dl = torch.utils.data.DataLoader(
        database_reader,
        batch_size=16,
        shuffle=False,
        collate_fn=database_reader.collate_fn,
        num_workers=4,
        pin_memory=True,
        drop_last=False,
    )
    
    # Query: single sample processing (no batching needed)
    query_reader = DataReader(
        csv_file=LOCAL_DATA_DIR / "query_lidar_frames.csv",
        lidar_scans_dir=SBER_OFFICE_DATA_DIR / "scans",
        pointcloud_quantization_size=PC_QUANTIZATION_SIZE,  # Must match database for consistency
        agg_thres=agg_thres,
    )

    eval_model(query_reader, database_dl, model)
    gc.collect()
    torch.cuda.empty_cache()

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 30.75it/s]


Descriptors shape: torch.Size([316, 256])


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1626/1626 [00:14<00:00, 111.40it/s]


Mean translation error: 6.70 m
Mean rotation error: 64.80 degrees
Median translation error: 4.42 m
Median rotation error: 45.62 degrees
Recall@1 at 5.0 m: 25.09%
Recall@2 at 5.0 m: 38.31%
Recall@3 at 5.0 m: 46.99%
Recall@4 at 5.0 m: 52.58%
Recall@5 at 5.0 m: 56.40%


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:09<00:00,  2.01it/s]


Descriptors shape: torch.Size([316, 256])


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1626/1626 [05:36<00:00,  4.83it/s]


Mean translation error: 12.88 m
Mean rotation error: 50.15 degrees
Median translation error: 11.55 m
Median rotation error: 38.21 degrees
Recall@1 at 5.0 m: 9.35%
Recall@2 at 5.0 m: 14.88%
Recall@3 at 5.0 m: 17.96%
Recall@4 at 5.0 m: 19.31%
Recall@5 at 5.0 m: 24.42%


 15%|█████████████████████████▉                                                                                                                                                   | 3/20 [00:04<00:26,  1.54s/it]


KeyboardInterrupt: 

## MinkLoc3Dv1 nclt

### Threshold = 0
```
Mean translation error: 4.76 m
Mean rotation error: 86.52 degrees
Median translation error: 2.29 m
Median rotation error: 89.87 degrees
Recall@1 at 5.0 m: 53.08%
Recall@2 at 5.0 m: 63.90%
Recall@3 at 5.0 m: 69.93%
Recall@4 at 5.0 m: 73.74%
Recall@5 at 5.0 m: 76.38%
```

### Threshold = 1
```
Mean translation error: 11.20 m
Mean rotation error: 67.64 degrees
Median translation error: 7.59 m
Median rotation error: 54.26 degrees
Recall@1 at 5.0 m: 26.08%
Recall@2 at 5.0 m: 33.03%
Recall@3 at 5.0 m: 37.64%
Recall@4 at 5.0 m: 42.62%
Recall@5 at 5.0 m: 46.25%
```

### Threshold = 2
```
Mean translation error: 10.77 m
Mean rotation error: 80.80 degrees
Median translation error: 5.41 m
Median rotation error: 70.28 degrees
Recall@1 at 5.0 m: 28.04%
Recall@2 at 5.0 m: 35.79%
Recall@3 at 5.0 m: 40.77%
Recall@4 at 5.0 m: 44.53%
Recall@5 at 5.0 m: 48.59%
```

### Threshold = 3
```
Mean translation error: 11.44 m
Mean rotation error: 82.51 degrees
Median translation error: 8.17 m
Median rotation error: 72.35 degrees
Recall@1 at 5.0 m: 29.83%
Recall@2 at 5.0 m: 35.18%
Recall@3 at 5.0 m: 39.79%
Recall@4 at 5.0 m: 42.87%
Recall@5 at 5.0 m: 44.71%
```

### Threshold = 4

```
Mean translation error: 12.37 m
Mean rotation error: 85.95 degrees
Median translation error: 10.54 m
Median rotation error: 81.24 degrees
Recall@1 at 5.0 m: 29.27%
Recall@2 at 5.0 m: 33.52%
Recall@3 at 5.0 m: 35.98%
Recall@4 at 5.0 m: 37.82%
Recall@5 at 5.0 m: 40.53%
```

## MinkLoc3Dv2 NCLT
### Threshold = 0
```
Mean translation error: 6.06 m
Mean rotation error: 75.90 degrees
Median translation error: 3.09 m
Median rotation error: 64.60 degrees
Recall@1 at 5.0 m: 37.88%
Recall@2 at 5.0 m: 50.92%
Recall@3 at 5.0 m: 56.21%
Recall@4 at 5.0 m: 60.82%
Recall@5 at 5.0 m: 64.94%
```

### Threshold = 1
```
Mean translation error: 8.78 m
Mean rotation error: 70.14 degrees
Median translation error: 4.72 m
Median rotation error: 54.23 degrees
Recall@1 at 5.0 m: 27.55%
Recall@2 at 5.0 m: 36.96%
Recall@3 at 5.0 m: 42.19%
Recall@4 at 5.0 m: 48.09%
Recall@5 at 5.0 m: 51.91%
```

### Threshold = 2
```
Mean translation error: 7.58 m
Mean rotation error: 84.03 degrees
Median translation error: 3.87 m
Median rotation error: 78.62 degrees
Recall@1 at 5.0 m: 35.18%
Recall@2 at 5.0 m: 46.99%
Recall@3 at 5.0 m: 53.38%
Recall@4 at 5.0 m: 57.13%
Recall@5 at 5.0 m: 60.89%
```

### Threshold = 3
```
Mean translation error: 8.19 m
Mean rotation error: 83.47 degrees
Median translation error: 4.37 m
Median rotation error: 78.65 degrees
Recall@1 at 5.0 m: 40.59%
Recall@2 at 5.0 m: 48.09%
Recall@3 at 5.0 m: 52.28%
Recall@4 at 5.0 m: 55.84%
Recall@5 at 5.0 m: 58.43%
```

### Threshold = 4
```
Mean translation error: 9.57 m
Mean rotation error: 87.32 degrees
Median translation error: 4.95 m
Median rotation error: 82.79 degrees
Recall@1 at 5.0 m: 31.92%
Recall@2 at 5.0 m: 39.30%
Recall@3 at 5.0 m: 43.67%
Recall@4 at 5.0 m: 46.86%
Recall@5 at 5.0 m: 50.49%
```

## MinkLoc3Dv2 Oxford
### Threshold = 0
```
Mean translation error: 4.89 m
Mean rotation error: 83.06 degrees
Median translation error: 2.63 m
Median rotation error: 82.46 degrees
Recall@1 at 5.0 m: 49.02%
Recall@2 at 5.0 m: 58.92%
Recall@3 at 5.0 m: 64.64%
Recall@4 at 5.0 m: 69.00%
Recall@5 at 5.0 m: 72.02%
```

### Threshold = 1
```
Mean translation error: 9.66 m
Mean rotation error: 71.00 degrees
Median translation error: 6.16 m
Median rotation error: 50.80 degrees
Recall@1 at 5.0 m: 24.91%
Recall@2 at 5.0 m: 32.35%
Recall@3 at 5.0 m: 38.62%
Recall@4 at 5.0 m: 42.74%
Recall@5 at 5.0 m: 46.25%
```

### Threshold = 2

```
Mean translation error: 10.51 m
Mean rotation error: 78.20 degrees
Median translation error: 7.41 m
Median rotation error: 60.79 degrees
Recall@1 at 5.0 m: 29.34%
Recall@2 at 5.0 m: 34.01%
Recall@3 at 5.0 m: 38.25%
Recall@4 at 5.0 m: 41.45%
Recall@5 at 5.0 m: 45.63%
```